# BanglaBERT fine-tuning on SentiGOLD (Bangla sentiment) + push to Hugging Face

**Kaggle setup (age ei 3 ta kore nao):**
1. Right panel → Settings → **Accelerator: GPU T4 x2 ba P100**, **Internet: On**
2. Add-ons → Secrets → `HF_TOKEN` naame tomar Hugging Face **write** token add koro
3. Niche `HF_REPO` e tomar username boshao

**Flow:** data load → normalize → split → tokenize → model → train → evaluate → push to Hub → test

In [1]:
!pip install -q git+https://github.com/csebuetnlp/normalizer

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 3.7 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 1. Config (sob settings ekhane)

In [2]:
import numpy as np, pandas as pd, torch
from transformers import set_seed

MODEL_NAME = "csebuetnlp/banglabert"
DATASET_ID = "SayedShaun/sentigold"
HF_REPO    = "sm-khalil/banglabert-sentigold"   #defined 

MAX_LEN = 128      # max tokens per text
BATCH   = 32
EPOCHS  = 3
LR      = 2e-5     # fine-tuning e choto LR use hoy, jate pretrained knowledge nosto na hoy
SEED    = 42

set_seed(SEED)
print("GPU available:", torch.cuda.is_available())

GPU available: True


## 2. Data load
SentiGOLD: ~70k Bangla text, multi-domain, 5 sentiment class (`Polarity` column). Label gulo string, tai amra integer id te convert korbo.

In [3]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID)["train"]
df = raw.to_pandas()[["Text", "Polarity"]].dropna()
df = df.drop_duplicates(subset="Text")          # duplicate thakle train/test leak hoy

print(df.shape)
print(df["Polarity"].value_counts())            # class imbalance ekhane dekho

labels   = sorted(df["Polarity"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["Polarity"].map(label2id)
print(label2id)
df.sample(5, random_state=SEED)

README.md:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.42MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

(70000, 2)
Polarity
SP    15318
NU    15156
SN    13767
WN    13682
WP    12077
Name: count, dtype: int64
{'NU': 0, 'SN': 1, 'SP': 2, 'WN': 3, 'WP': 4}


,Text,Polarity,label
46730,সবই ঠিক আছে কিন্তু আপনার গলার হারের লকেট থেকে ...,WN,3
48393,এই সময় ব্যাগের মালিক মুক্তারকে আটক করা হয়।,WN,3
41416,আমি এই ভিডিওটি দেখতেই শান্তি এবং একটি আকাঙ্খা ...,SP,2
34506,২০১৪ সাল একটি মাইলফলক হয়ে থাকবে বাংলাদেশের জন...,SP,2
43725,বোধকরি জীবজগতে একমাত্র মানুষেরই ধর্ম(প্রাতিষ্ঠ...,SP,2


## 3. Normalize + split
BanglaBERT er authors er `normalizer` use na korle model er performance kome. **Inference e o same normalize korte hobe.**
Split: 80% train / 10% validation / 10% test, stratified (protita class er ratio same thake).

In [4]:
from normalizer import normalize
from sklearn.model_selection import train_test_split

df["text"] = df["Text"].astype(str).apply(normalize)
df = df[df["text"].str.strip().str.len() > 0]

train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

56000 7000 7000


## 4. Tokenize
Model text bujhe na, token id bujhe. `DataCollatorWithPadding` protita batch er sobcheye lomba text er length porjonto pad kore (dynamic padding), tai GPU time bache.

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(d):
    ds = Dataset.from_pandas(d[["text", "label"]].reset_index(drop=True))
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text"])
    return ds.rename_column("label", "labels")

train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)
collator = DataCollatorWithPadding(tokenizer)

# ekta sample er tokenization dekho
s = train_df["text"].iloc[0]
print(s)
print(tokenizer.tokenize(s))

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/56000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

সাগর সেচে কুড়িয়ে পাওয়া মানিক আমার নাবিলা।
['সাগর', 'সেচ', '##ে', 'কুড়িয়ে', 'পাওয়া', 'মানিক', 'আমার', 'নাবিল', '##া', '।']


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 5. Model
`AutoModelForSequenceClassification` = pretrained BanglaBERT body + ekta notun, random-initialized classification head (`num_labels` ta output).
Fine-tuning mane: body ar head, dutor weight-i ekshathe update hoy (full fine-tuning).

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, "M")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 110.621189 M


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

## 6. (Learning) Ekta training step haate haate
`Trainer` ashole ei kaj-i bar bar kore: forward → loss → backward → optimizer step. Ekbar eta bujhe nao.

In [8]:
batch = collator([train_ds[i] for i in range(8)])
out = model(**batch)                  # forward pass; labels thakle loss o return kore
print("loss:", out.loss.item(), "| logits shape:", tuple(out.logits.shape))

out.loss.backward()                   # backward: protita weight er gradient toiri hoy
g = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters() if p.grad is not None))
print("gradient norm:", g.item())

# optimizer.step() ei gradient diye weight update kore: w = w - lr * (adjusted gradient)
model.zero_grad()                     # porer step er age gradient clear

loss: 1.6111087799072266 | logits shape: (8, 5)
gradient norm: 1.7150286436080933


## 7. Baseline (TF-IDF + Logistic Regression)
Deep model er result er sathe compare korar jonno. CV te "X vs baseline Y" likhte kaje lagbe.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000)
Xtr = tfidf.fit_transform(train_df["text"])
Xte = tfidf.transform(test_df["text"])
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, train_df["label"])
bp = lr.predict(Xte)
print("Baseline  acc:", accuracy_score(test_df["label"], bp),
      "| macro-F1:", f1_score(test_df["label"], bp, average="macro"))

Baseline  acc: 0.397 | macro-F1: 0.3921915249599415


## 8. Train
- `warmup_ratio=0.1`: shuru te LR dhire dhire barano hoy, training stable thake
- `weight_decay`: overfitting kombhuto kore
- `fp16`: GPU te 2x fast ar kom memory
- Best epoch (validation macro-F1 onujayi) er model automatically load hoy

In [11]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(y, preds),
            "f1_macro": f1_score(y, preds, average="macro")}

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class=tokenizer,       # purono transformers e error dile eta `tokenizer=tokenizer` koro
    compute_metrics=compute_metrics,
)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.748124,1.709404,0.655286,0.646813
2,1.472654,1.683761,0.661429,0.658067
3,1.297505,1.771679,0.667143,0.662338


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2625, training_loss=1.5902215038481213, metrics={'train_runtime': 880.102, 'train_samples_per_second': 190.887, 'train_steps_per_second': 2.983, 'total_flos': 3330516557541504.0, 'train_loss': 1.5902215038481213, 'epoch': 3.0})

## 9. Test set e final evaluation
Test set training/validation e kokhono dekha hoyni, tai eta-i tomar asol result.

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(test_ds)
y_pred, y_true = pred.predictions.argmax(-1), pred.label_ids

print(classification_report(y_true, y_pred, target_names=labels, digits=3))
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=labels, columns=labels))

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

          NU      0.714     0.695     0.704      1516
          SN      0.704     0.720     0.712      1376
          SP      0.721     0.763     0.741      1532
          WN      0.612     0.602     0.607      1368
          WP      0.587     0.558     0.572      1208

    accuracy                          0.673      7000
   macro avg      0.667     0.668     0.667      7000
weighted avg      0.672     0.673     0.672      7000

      NU   SN    SP   WN   WP
NU  1054   49    79  152  182
SN    47  991    46  276   16
SP    82   32  1169   28  221
WN   149  313    28  823   55
WP   145   23   300   66  674


## 10. Error analysis
Model kothay bhul korche dekho. README te 5-10 ta example likho.

In [13]:
t = test_df.reset_index(drop=True).copy()
t["pred"] = [id2label[i] for i in y_pred]
wrong = t[t["Polarity"] != t["pred"]]
print("Wrong:", len(wrong), "/", len(t))
wrong[["text", "Polarity", "pred"]].sample(10, random_state=SEED)

Wrong: 2289 / 7000


,text,Polarity,pred
6741,আমরা এইচ এস সি পরাক্ষার্থী না শুধু আমাদের মধ্য...,SP,WP
1563,"জলদাসের প্রথম উপন্যাস এই জলপুত্র, প্রথম উপন্যা...",WP,SP
2599,খালেদা জিয়াও তাদের হাত নেড়ে জবাব দেন।,WP,NU
129,মৌলভীবাজারের গ্র ্যান্ড সুলতান একটা ইত্যাদি চাই।,WP,NU
6095,বিষয়বস্তুর দিকে লক্ষ রেখে আমরা এর বাংলা নাম দ...,SP,NU
821,আমাদেরও জনগণের সামনে মুখ রক্ষা হলো আর যুক্তরাষ...,NU,WP
3226,আরও দুইজন শুভ্র এবং মিসবাহ যথাক্রমে প্রোমোশন এ...,NU,WP
6074,সুলতানা আক্তার টাকা লাগবে তবেই এড দেবো,WN,NU
2920,"২৭ অক্টোবর, ২০০১: নওগাঁয় পূজামণ্ডপে আগুন, কুল...",WN,SN
1040,লক্ষ মানুষের ভিড়ের হট্টগোলেও নেমে আসে পিনপতন ...,SN,WN


## 11. Hugging Face e upload
Prothome HF e login, tarpor model + tokenizer push. `private=True` rakhle shudhu tumi dekhbe (token lagbe). Public korte chaile `private=False`.

In [14]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

trainer.model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)
print("Pushed to https://huggingface.co/" + HF_REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed to https://huggingface.co/sm-khalil/banglabert-sentigold


## 12. Hub theke load kore test (tomar project e ei code-i boshbe)

In [15]:
from transformers import pipeline
from normalizer import normalize

clf = pipeline("text-classification", model=HF_REPO, device=0 if torch.cuda.is_available() else -1)

def predict(text: str):
    return clf(normalize(text))[0]      # {'label': ..., 'score': ...}

for s in ["বইটা অসাধারণ, পড়ে খুব ভালো লাগলো", "একদম বাজে সার্ভিস, আর কখনো আসবো না"]:
    print(s, "->", predict(s))

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

বইটা অসাধারণ, পড়ে খুব ভালো লাগলো -> {'label': 'SP', 'score': 0.9810318350791931}
একদম বাজে সার্ভিস, আর কখনো আসবো না -> {'label': 'SN', 'score': 0.9633169174194336}
